# Private LoRA16 completion from verified step 804
Attach original private Task1 data and pa3-lora16-step804-restore.zip privately. Previous run must be terminal. Resume all446 remaining updates to1250 in one saved background job,10800s safety timeout. Exact training recipe unchanged. Download and verify actual final archive. Historical312/625 checkpoints remain on the original Modal volume; this input preserves804 only.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'


In [ ]:
import pathlib, json
work = pathlib.Path("/kaggle/working")
def restore_verified_inputs(input_root, destination, checkpoint_name, expected):
    import hashlib
    from pathlib import Path, PurePosixPath
    import shutil
    import zipfile

    input_root, destination = Path(input_root), Path(destination)
    if destination.exists():
        raise FileExistsError("Refusing to overwrite an existing recovery directory")
    relative = {}
    for name, digest in expected.items():
        parts = PurePosixPath(name).parts
        if len(parts) < 2 or ".." in parts or PurePosixPath(name).is_absolute():
            raise ValueError("Unsafe checkpoint manifest path")
        relative[str(PurePosixPath(*parts[1:]))] = digest
    marker = checkpoint_name + "/complete.json"
    if marker not in relative:
        raise ValueError("Checkpoint manifest has no complete marker")
    candidates = [("directory", p.parent.parent, "")
                  for p in sorted(input_root.rglob("complete.json"))
                  if p.parent.name == checkpoint_name]
    for archive in sorted(input_root.rglob("*.zip")):
        with zipfile.ZipFile(archive) as source:
            for name in source.namelist():
                if name == marker or name.endswith("/" + marker):
                    candidates.append(("zip", archive, name[:-len(marker)]))
    if not candidates:
        raise ValueError(f"No extracted or zipped complete {checkpoint_name} input found")
    # Every duplicate must match the entire pinned manifest before any write.
    for kind, location, prefix in candidates:
        if kind == "directory":
            for name, digest in relative.items():
                file = location / name
                if not file.is_file():
                    raise ValueError(f"Checkpoint file missing: {file}")
                with file.open("rb") as stream:
                    actual = hashlib.file_digest(stream, "sha256").hexdigest()
                if actual != digest:
                    raise ValueError(f"Checkpoint checksum mismatch: {file}")
        else:
            with zipfile.ZipFile(location) as source:
                for name, digest in relative.items():
                    with source.open(prefix + name) as stream:
                        actual = hashlib.file_digest(stream, "sha256").hexdigest()
                    if actual != digest:
                        raise ValueError(f"Checkpoint checksum mismatch: {location}:{prefix + name}")
    kind, location, prefix = candidates[0]
    destination.mkdir(parents=True)
    # Copy only pinned files; complete markers come after their recovery payloads.
    order = sorted(relative, key=lambda name: (name.endswith("/complete.json"), name))
    if kind == "directory":
        for name in order:
            target = destination / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with (location / name).open("rb") as source, target.open("xb") as output:
                shutil.copyfileobj(source, output)
    else:
        with zipfile.ZipFile(location) as source:
            for name in order:
                target = destination / name
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(prefix + name) as stream, target.open("xb") as output:
                    shutil.copyfileobj(stream, output)
    print("VERIFIED_INPUT_COPIES", len(candidates), flush=True)
    return destination / checkpoint_name

EXPECTED = {'lora16-recovery/sessions.jsonl': 'a27ce60251b72e661f0aaa29476ac7675240772421a714e0797f6fba7e14048c', 'lora16-recovery/status.json': '8fb82a2a749f258f7e21a11ef85162c1b9e4d85d51a420ca9a320d5bd2b5b791', 'lora16-recovery/train.jsonl': 'a2f531393990762e121bcfe6d9e5427bbcb122acde52561ad6646cd362758ceb', 'lora16-recovery/resume-0804-1793464816710527703/complete.json': '855c1dd920475a4184e6a20b1eaa923448168fe87c24c1c5ee541b94c19f0e49', 'lora16-recovery/resume-0804-1793464816710527703/recovery.pt': '7c3d4cb486a3f09c638fad8dfbee7cdebbb016414050b1da8b86a51d2d11d992', 'lora16-recovery/resume-0804-1793464816710527703/summary.json': 'ea54c3a4422f82d4a1ab8c3855b9592eada97d85c3456528371233a4df7892e5', 'lora16-recovery/resume-0804-1793464816710527703/projected_factors.pt': '03c288e53597b0fe69a15e664f7b46605dd20afb0f9579a733fece65debfbbb9'}
run = work / 'pa3/runs/lora16-20261030'
checkpoint = restore_verified_inputs('/kaggle/input', run, 'resume-0804-1793464816710527703', EXPECTED)
summary = json.loads((checkpoint / 'summary.json').read_text())
assert summary['update'] == 804 and summary['successful_optimizer_updates'] == 804 and summary['skipped_updates'] == 0
print('VERIFIED_RESTORE', summary, flush=True)


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "cf7f042114a0a924403ed55ef9ee5982946ecf38"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)


In [ ]:
# Verify all original input bytes; tolerate only byte-identical duplicate copies.
import sys, pathlib, zipfile, hashlib
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs, select_original_input
EXPECTED_TRAIN_SHA = '644a46f9ab7a8d368499b2054229cc7324614d97596fbdee1ac3ab5cbb584de9'
archives = list(pathlib.Path('/kaggle/input').rglob('pa3-task1-input.zip'))
if len(archives) == 1:
    input_archive = archives[0]
else:
    original = select_original_input('/kaggle/input', EXPECTED_TRAIN_SHA)
    input_archive = work / 'pa3-task1-input.zip'
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, 'x', zipfile.ZIP_DEFLATED) as z:
            for name in ('train.jsonl', 'metadata.json', 'source_ids.json'):
                z.write(original / name, 'task1/' + name)
data_dir = unpack_inputs(input_archive, work / 'pa3/data')
assert hashlib.file_digest((data_dir / 'train.jsonl').open('rb'), 'sha256').hexdigest() == EXPECTED_TRAIN_SHA
print('Original training data checksum verified:', data_dir, flush=True)


In [ ]:
import subprocess, sys, json, zipfile, hashlib
command = [sys.executable, '-m', 'scripts.kaggle_task1', '--archive', str(input_archive), '--method', 'lora16', '--run-id', 'lora16-20261030', '--seconds', '10800', '--max-updates', '446', '--resume', str(checkpoint), '--handoff-verified']
print('START_BOUNDED_RECOVERY_FROM', checkpoint, flush=True)
try:
    subprocess.check_call(command, cwd=source)
finally:
    status = json.loads((run / 'status.json').read_text())
    saved = run / pathlib.Path(status['checkpoint']).name
    assert (saved / 'complete.json').is_file()
    result = json.loads((saved / 'summary.json').read_text())
    assert 804 <= result['update'] <= 1250
    archive = work / ('lora16-backup-step-%04d.zip' % result['update'])
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as backup:
        for file in run.rglob('*'):
            if file.is_file():
                backup.write(file, str(pathlib.Path('lora16-recovery') / file.relative_to(run)))
    with zipfile.ZipFile(archive) as backup:
        assert backup.testzip() is None
    evidence = {'status': status, 'summary': result, 'archive': str(archive), 'archive_bytes': archive.stat().st_size, 'archive_sha256': hashlib.file_digest(archive.open('rb'), 'sha256').hexdigest()}
    (work / 'checkpoint-backup-evidence.json').write_text(json.dumps(evidence, indent=2))
    print('DOWNLOAD_AND_VERIFY_BEFORE_NEXT_RUN', json.dumps(evidence), flush=True)
